# 1000 Soils – Random Forest Workshop

This notebook builds a **Random Forest regression** model to predict soil respiration (`respiration_ppm_co2_c`) from biogeochemical and site metadata features collected across the 1000 Soils dataset.

**Workshop workflow:**
1. **Exploratory Data Analysis (EDA)** — understand the target variable and its relationships with predictors
2. **Data Gaps** — identify and handle missing data systematically
3. **Train/Test Split** — create a representative split and verify it covers the full distribution
4. **Model Performance** — train a baseline Random Forest and evaluate its predictions
5. **Hyperparameter Tuning** *(optional)* — improve model performance with systematic search
6. **SHAP Analysis** — explain *why* the model makes its predictions

In [ ]:
import pandas as pd
import os
import subprocess
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [ ]:
!pip install zenodo-get

In [ ]:
# Download data from Zenodo
cmd = ["zenodo_get", "10.5281/zenodo.15328215"]
working_directory = "1000Soil_data"
os.makedirs(working_directory, exist_ok=True)

process = subprocess.Popen(
    cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, cwd=working_directory
)
output, error = process.communicate()

if error:
    print("An error occurred:", error.decode())
else:
    print("Download complete!")
    print(output.decode())

## Data Loading & Preprocessing

Loads two data sources and merges them on `Site`:
- **`1000S_processed_BGC_summary.csv`** — biogeochemical measurements (pH, C/N, moisture, etc.) per soil core section.
- **`1000Soils_Metadata_Site_Mastersheet_v1.xlsx`** — site-level metadata (coordinates, land cover, etc.).

In [ ]:
# Load metadata
metadata = pd.read_excel("1000Soil_data/1000Soils_Metadata_Site_Mastersheet_v1.xlsx")
metadata['Site'] = metadata['Site_Code'].str.replace(' ', '')

# Load biogeochemical data
df_BG = pd.read_csv("1000Soil_data/1000S_processed_BGC_summary.csv")
print(f"BGC rows: {len(df_BG)}")

# Extract site code and create unique sample identifier
df_BG.insert(1, 'Site', df_BG['Site_ID'].str.split('_').str[1])
df_BG['Sample'] = df_BG['Site'] + "_" + df_BG['Core_Section']

# Merge with metadata
df_BG = df_BG.merge(metadata, how='outer', on='Site')

# Fix missing coordinates
wsu_rows = df_BG['Site'].str.contains("WSU", case=False, na=False)
df_BG.loc[wsu_rows,                     ['Lat', 'Long']] = [46.2523,    -119.737]
df_BG.loc[df_BG['Site'].isin(['OAES']), ['Lat', 'Long']] = [35.410599,  -99.058779]
df_BG.loc[df_BG['Site'].isin(['TEAK']), ['Lat', 'Long']] = [37.00583,  -119.00602]

# Biome mapping
biome_dict = {
    'ANZA': 'Desert shrubland',   'FTA3': 'Desert shrubland',
    'FTA5': 'Desert shrubland',   'JORN': 'Desert shrubland',
    'MOAB': 'Desert shrubland',   'OCTB': 'Desert shrubland',
    'OCTU': 'Desert shrubland',   'ONAQ': 'Desert shrubland',
    'PRS2': 'Desert shrubland',   'SRER': 'Desert shrubland',
    'SRR1': 'Desert shrubland',   'UT32': 'Desert shrubland',
    'WSU1': 'Desert shrubland',   'WSU2': 'Desert shrubland',
    'WSU3': 'Desert shrubland',   'OAES': 'Desert shrubland',
    'SJER': 'Mediterranean woodlands',
    'BLAN': 'Temperate forests',  'CFS2': 'Temperate forests',
    'DELA': 'Temperate forests',  'GRSM': 'Temperate forests',
    'LENO': 'Temperate forests',  'MLBS': 'Temperate forests',
    'NWBA': 'Temperate forests',  'NWBB': 'Temperate forests',
    'NWBC': 'Temperate forests',  'ORNL': 'Temperate forests',
    'PPRH': 'Temperate forests',  'PRS1': 'Temperate forests',
    'SCBI': 'Temperate forests',  'SERC': 'Temperate forests',
    'TALL': 'Temperate forests',  'WLLO': 'Temperate forests',
    'WLUP': 'Temperate forests',
    'CFS1': 'Temperate coniferous forests', 'DSNY': 'Temperate coniferous forests',
    'JERC': 'Temperate coniferous forests', 'OKPF': 'Temperate coniferous forests',
    'OSBS': 'Temperate coniferous forests', 'PETF': 'Temperate coniferous forests',
    'PHTU': 'Temperate coniferous forests', 'RMNP': 'Temperate coniferous forests',
    'SOAP': 'Temperate coniferous forests', 'UT12': 'Temperate coniferous forests',
    'UT19': 'Temperate coniferous forests', 'UT23': 'Temperate coniferous forests',
    'UT24': 'Temperate coniferous forests', 'WY01': 'Temperate coniferous forests',
    'WY03': 'Temperate coniferous forests', 'WY09': 'Temperate coniferous forests',
    'WY10': 'Temperate coniferous forests', 'WY15': 'Temperate coniferous forests',
    'TEAK': 'Temperate coniferous forests',
    'CLBJ': 'Temperate grasslands', 'CPER': 'Temperate grasslands',
    'DCFS': 'Temperate grasslands', 'ISCC': 'Temperate grasslands',
    'ISNC': 'Temperate grasslands', 'KONA': 'Temperate grasslands',
    'KONZ': 'Temperate grasslands', 'NOGP': 'Temperate grasslands',
    'UKFS': 'Temperate grasslands', 'STER': 'Temperate grasslands',
}

df_BG['BiomeType'] = df_BG['Site'].map(biome_dict)
df_BG['Sample']    = df_BG['Site'] + "_" + df_BG['Core_Section']
df_BG = df_BG.rename(columns={'Site_ID': 'Sample_ID'})

df_merged = df_BG
print(f"Merged rows: {len(df_merged)}")
df_merged.head(5)

In [ ]:
from sklearn.preprocessing import LabelEncoder

cat_cols = ['BiomeType', 'Weather', 'Vegetation', 'Site']
le = LabelEncoder()
for col in cat_cols:
    enc_col  = f'{col}_enc'
    non_null = df_merged[col].notna()
    df_merged[enc_col] = -1
    df_merged.loc[non_null, enc_col] = le.fit_transform(df_merged.loc[non_null, col])

print("Encoded columns added:")
for col in cat_cols:
    n = df_merged[f'{col}_enc'].nunique()
    print(f"  {col}_enc : {n} unique codes")

---
## Step 1: Exploratory Data Analysis (EDA)

Before building any model, it is essential to **understand the data**:
- What does the target variable look like? Is it normally distributed or skewed?
- How do predictors relate to the target?
- Are there obvious patterns or outliers?

> **Why this matters:** EDA informs feature engineering decisions, reveals potential data quality issues, and builds intuition about what drives soil respiration.

In [ ]:
target = 'respiration_ppm_co2_c'

# Filter to rows where the target is measured
df_target = df_merged.dropna(subset=[target])
print(f"Total rows in merged dataset : {len(df_merged)}")
print(f"Rows with target measured    : {len(df_target)}")
print()
df_target[target].describe().to_frame().T

### 1.1 Target Variable Distribution

Plot the raw distribution **and** the log-transformed distribution.
A right-skewed target (common for flux data) may benefit from log-transformation,
but Random Forests are generally robust to skewness.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df_target[target], bins=40, color='steelblue', edgecolor='k', linewidth=0.4)
axes[0].set_xlabel('Respiration (ppm CO₂-C)')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of Soil Respiration\n(raw)')

axes[1].hist(np.log1p(df_target[target]), bins=40, color='coral', edgecolor='k', linewidth=0.4)
axes[1].set_xlabel('log(1 + Respiration)')
axes[1].set_ylabel('Count')
axes[1].set_title('Distribution of Soil Respiration\n(log-transformed)')

for ax in axes:
    ax.spines[['top', 'right']].set_visible(False)
plt.suptitle('Target Variable: Soil Respiration', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"Skewness: {df_target[target].skew():.2f}  (|skew| > 1 suggests heavy right-skew)")

### 1.2 Respiration vs Top Correlated Predictors

Compute the Pearson correlation of every numeric column with the target and plot the top 9.
Scatter plots reveal:
- **Linear vs non-linear** relationships
- **Outliers** that may influence the model
- **Spread** — high spread means the predictor alone explains little variance

In [ ]:
# Exclude direct leakage columns (other respiration measurements in different units)
exclude_from_eda = [
    target,
    'respiration_mg_co2_c_g_soil_day_96hour',
    'respiration_mg_co2_c_g_soil_day_24hour',
    'VolWaterContent.m3/m3',
]
numeric_cols = [
    c for c in df_target.select_dtypes(include=np.number).columns
    if c not in exclude_from_eda
]

# Rank by absolute Pearson correlation with the target
corrs = df_target[numeric_cols + [target]].corr()[target].drop(target)
top_predictors = corrs.abs().sort_values(ascending=False).head(9).index.tolist()

fig, axes = plt.subplots(3, 3, figsize=(14, 12))
for ax, col in zip(axes.flatten(), top_predictors):
    r = df_target[[col, target]].dropna().corr().iloc[0, 1]
    ax.scatter(df_target[col], df_target[target],
               alpha=0.4, s=15, edgecolors='none', color='steelblue')
    ax.set_xlabel(col, fontsize=9)
    ax.set_ylabel('Respiration', fontsize=9)
    ax.set_title(f'{col}\nr = {r:.2f}', fontsize=9)
    ax.spines[['top', 'right']].set_visible(False)

plt.suptitle('Soil Respiration vs Top 9 Correlated Predictors',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### 1.3 Respiration by Biome Type

Boxplots show the **distribution within each biome**, sorted by median respiration.
This reveals whether ecosystem type is a strong driver — and whether some biomes
show much greater variability than others.

In [ ]:
df_biome = df_target.dropna(subset=['BiomeType'])
biome_order = (
    df_biome.groupby('BiomeType')[target]
    .median()
    .sort_values(ascending=False)
    .index.tolist()
)

fig, ax = plt.subplots(figsize=(13, 5))
bp_data = [df_biome.loc[df_biome['BiomeType'] == b, target].values for b in biome_order]
bp = ax.boxplot(bp_data, labels=biome_order, patch_artist=True,
                medianprops=dict(color='black', linewidth=2))

colors = plt.cm.Set2(np.linspace(0, 1, len(biome_order)))
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)

ax.set_xticklabels(biome_order, rotation=30, ha='right')
ax.set_ylabel('Respiration (ppm CO₂-C)')
ax.set_title('Soil Respiration by Biome Type (sorted by median)', fontweight='bold')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

---
## Step 2: Understanding Data Gaps

Real-world soil datasets are rarely complete — instruments fail, samples are too small,
or certain assays were not run at every site.

Understanding **where data is missing** is critical before feature selection:
- Including a feature with 80 % missingness would drastically shrink the usable sample size
- Columns that are mostly empty should be dropped before modelling

**Strategy used here:**
- Features with **> 30 % missing values** (among target rows) are excluded from the model
- Rows still missing values after feature selection are dropped (complete-case analysis)

In [ ]:
missing_summary = pd.DataFrame({
    'Missing Count': df_target.isna().sum(),
    'Missing %'    : (df_target.isna().mean() * 100).round(1),
}).sort_values('Missing %', ascending=False)
missing_summary = missing_summary[missing_summary['Missing Count'] > 0]

print(f"Total rows (target non-null) : {len(df_target)}")
print(f"Columns with any missing data: {len(missing_summary)}")
display(missing_summary.head(25))

In [ ]:
top_missing = missing_summary.head(25)
bar_colors  = ['#d62728' if p > 30 else '#aec7e8' for p in top_missing['Missing %']]

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(top_missing.index[::-1], top_missing['Missing %'][::-1],
        color=bar_colors[::-1], edgecolor='k', linewidth=0.4)
ax.axvline(30, color='red', linestyle='--', linewidth=1.5, label='30 % exclusion threshold')
ax.set_xlabel('Missing Data (%)')
ax.set_title('Top 25 Columns by Missing Data\n(red bars will be excluded from the model)',
             fontweight='bold')
ax.legend()
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

---
## Step 3: Feature Selection & Train/Test Split

### Feature Selection
We select features by:
1. Excluding identifier columns, free-text fields, and variables that directly measure the target (data leakage)
2. Dropping encoded categoricals with low expected predictive value for this demo (`Vegetation_enc`, `Weather_enc`, `Site_enc`), while keeping `BiomeType_enc`
3. Applying a **≤ 30 % missing data threshold** — any feature with more than 30 % NaN is dropped
4. Calling `.dropna()` on the final feature + target subset for a complete-case dataset

### Train/Test Split
- An **80/20 split** reserves 20 % of samples for final evaluation
- The test set must **never** be used during training or hyperparameter tuning
- We verify the split is representative by comparing the target distribution in both sets

In [ ]:
# Columns to always exclude from the model
drop_cols = [
    'Sample', 'Site_Code', 'Site_Name_Long', 'Core_collector',
    'DateTime_Collected (24 Hr format)',
    'ROI.volume.xyz', 'Location',
    'respiration_mg_co2_c_g_soil_day_96hour',  # other respiration unit (leakage)
    'respiration_mg_co2_c_g_soil_day_24hour',  # other respiration unit (leakage)
    'VolWaterContent.m3/m3',
    'Vegetation_enc',  # dropped for demo
    'Weather_enc',     # dropped for demo
    'Site_enc',        # dropped for demo
]

# Candidate numeric columns
candidate_cols = [
    c for c in df_target.columns
    if c != target
    and c not in drop_cols
    and pd.api.types.is_numeric_dtype(df_target[c])
]

# Keep only columns with <= 30 % missing among target rows
missing_pct  = df_target[candidate_cols].isna().mean()
feature_cols = missing_pct[missing_pct <= 0.30].index.tolist()
dropped      = [c for c in candidate_cols if c not in feature_cols]

print(f"Feature columns kept          : {len(feature_cols)}")
print(f"Dropped (> 30 % missing)      : {len(dropped)}")
if dropped:
    print("  Dropped:", dropped)

# Complete-case subset (drop rows with any remaining NaN)
df_ml = df_target[[target] + feature_cols].copy().dropna()
print(f"Complete-case rows for ML     : {len(df_ml)}")

In [ ]:
X = df_ml[feature_cols]
y = np.log1p(df_ml[target])

print(f"Samples : {len(df_ml)}")
print(f"Features: {len(feature_cols)}")
print("\nFeature columns:")
for i, col in enumerate(feature_cols, 1):
    print(f"  {i:2d}. {col}")

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training samples : {len(X_train)}")
print(f"Test samples     : {len(X_test)}")

### 3.1 Verify Split Representativeness

The train and test distributions should **overlap closely**.
A large mismatch would mean the model is evaluated on out-of-distribution data,
making test performance unreliable.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(y_train, bins=30, alpha=0.6, color='steelblue', density=True,
        edgecolor='k', linewidth=0.3, label=f'Train  (n={len(y_train)})')
ax.hist(y_test,  bins=30, alpha=0.6, color='coral',     density=True,
        edgecolor='k', linewidth=0.3, label=f'Test   (n={len(y_test)})')
ax.set_xlabel('Respiration (ppm CO₂-C)')
ax.set_ylabel('Density')
ax.set_title(
    'Target Distribution: Train vs Test\n'
    '(overlapping distributions confirm a representative split)',
    fontweight='bold',
)
ax.legend(framealpha=0.9)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

---
## Step 4: Random Forest Model — Baseline Performance

We start with a **lightly configured Random Forest** to establish a baseline before any
systematic hyperparameter search.

| Parameter | Value | Meaning |
|-----------|-------|---------|
| `n_estimators` | 100 | Number of decision trees in the forest |
| `max_features` | `'sqrt'` | Features considered at each split (reduces tree correlation) |
| `min_samples_leaf` | 2 | Minimum samples required at each leaf (regularisation) |
| `bootstrap` | `True` | Each tree trained on a bootstrap sample of the training data |

### What to look for
- **Training R²** is almost always higher than **Test R²** — this is expected (the model has memorised some training noise)
- A **large gap** (e.g. train R² = 0.95, test R² = 0.40) suggests overfitting
- **RMSE** is in the original target units — it is the average prediction error

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error

rf = RandomForestRegressor(
    n_estimators=100,
    max_features='sqrt',
    min_samples_leaf=2,
    bootstrap=True,
    random_state=42,
    n_jobs=-1,
)
rf.fit(X_train, y_train)

y_pred_train = rf.predict(X_train)
y_pred_test  = rf.predict(X_test)

r2_train   = r2_score(y_train, y_pred_train)
r2_test    = r2_score(y_test,  y_pred_test)
rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))
rmse_test  = np.sqrt(mean_squared_error(y_test,  y_pred_test))

print("── Baseline Random Forest ──────────────────────────────")
print(f"  Training   R²  : {r2_train:.3f}")
print(f"  Training   RMSE: {rmse_train:.4f}")
print(f"  Test       R²  : {r2_test:.3f}")
print(f"  Test       RMSE: {rmse_test:.4f}")

### 4.1 Predicted vs Actual

Plot both the **training** and **test** set predictions on a scatter plot.
Points close to the **1:1 dashed line** indicate accurate predictions.

- **Training plot** shows how well the model memorised the training data
- **Test plot** shows how well it *generalises* to unseen samples — this is the important one

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, y_true, y_pred, label, r2, rmse, color in [
    (axes[0], y_train, y_pred_train, 'Training', r2_train, rmse_train, 'steelblue'),
    (axes[1], y_test,  y_pred_test,  'Test',     r2_test,  rmse_test,  'coral'),
]:
    lims = [
        min(float(y_true.min()), float(y_pred.min())),
        max(float(y_true.max()), float(y_pred.max())),
    ]
    ax.scatter(y_true, y_pred, alpha=0.6, edgecolors='k', linewidths=0.3, s=30, color=color)
    ax.plot(lims, lims, 'k--', lw=1.5, label='Perfect prediction (1:1)')
    ax.set_xlabel('Actual Respiration (ppm CO₂-C)')
    ax.set_ylabel('Predicted Respiration')
    ax.set_title(f'{label} Set\nR² = {r2:.3f}  |  RMSE = {rmse:.4f}')
    ax.legend(fontsize=8)
    ax.spines[['top', 'right']].set_visible(False)

plt.suptitle('Random Forest — Predicted vs Actual (Baseline)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 4.2 Feature Importance

The bar chart below shows **Mean Decrease in Impurity (MDI)** — how much each feature
reduces variance across all splits in all trees, averaged and normalised to sum to 1.

**What it tells you:**
- Features at the **top** are most influential in the model's splitting decisions
- Features near the **bottom** contribute little predictive power

**Important caveats:**
- MDI shows *magnitude*, not *direction* — a feature can increase or decrease the prediction depending on its value (for directionality, see SHAP in Step 6)
- MDI can be slightly biased toward high-cardinality (many unique values) features
- Importance is relative to *this model* and does not imply causation

In [ ]:
importances = pd.Series(rf.feature_importances_, index=feature_cols)
top20 = importances.nlargest(20).sort_values()

fig, ax = plt.subplots(figsize=(9, 7))
top20.plot(kind='barh', ax=ax, color='steelblue', edgecolor='k', linewidth=0.4)
ax.set_xlabel('Feature Importance (Mean Decrease in Impurity)')
ax.set_title('Top 20 Feature Importances — Baseline Random Forest', fontweight='bold')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

---
## Step 5 (Optional): Hyperparameter Tuning

> **We may skip this section during the tutorial if time is short. The code is provided for reference.**

By default, Random Forest works reasonably well out-of-the-box. However, systematic
hyperparameter tuning can reduce overfitting and improve test set performance.

### Key hyperparameters to tune

| Parameter | Effect |
|-----------|--------|
| `n_estimators` | More trees → more stable predictions, but diminishing returns after ~300 |
| `max_depth` | Shallower trees reduce overfitting |
| `min_samples_leaf` | Higher values act as regularisation |
| `max_features` | Lower fraction increases diversity between trees |

We use **`RandomizedSearchCV`** which randomly samples combinations from a parameter grid
and evaluates each via **5-fold cross-validation**. Much faster than exhaustive grid search.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    'n_estimators'    : [100, 200, 300, 500, 700],
    'max_depth'       : [5, 10, 20],
    'min_samples_leaf': [1, 2, 4, 8, 16],
    'max_features'    : ['sqrt', 0.5],
}

search = RandomizedSearchCV(
    RandomForestRegressor(bootstrap=True, random_state=42, n_jobs=-1),
    param_distributions=param_dist,
    n_iter=50,
    cv=5,
    scoring='r2',
    random_state=42,
    n_jobs=-1,
    verbose=1,
)
search.fit(X_train, y_train)

print(f"\nBest parameters : {search.best_params_}")
print(f"Best CV R²      : {search.best_score_:.3f}")

In [ ]:
rf_tuned = search.best_estimator_

y_pred_tuned = rf_tuned.predict(X_test)
r2_tuned     = r2_score(y_test, y_pred_tuned)
rmse_tuned   = np.sqrt(mean_squared_error(y_test, y_pred_tuned))

print("── Performance Comparison (Test Set) ──────────────────")
print(f"  Baseline  R²  : {r2_test:.3f}   RMSE: {rmse_test:.4f}")
print(f"  Tuned     R²  : {r2_tuned:.3f}   RMSE: {rmse_tuned:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, y_pred, label, r2, rmse, color in [
    (axes[0], y_pred_test,  'Baseline', r2_test,  rmse_test,  'steelblue'),
    (axes[1], y_pred_tuned, 'Tuned',    r2_tuned, rmse_tuned, 'mediumpurple'),
]:
    lims = [
        min(float(y_test.min()), float(y_pred.min())),
        max(float(y_test.max()), float(y_pred.max())),
    ]
    ax.scatter(y_test, y_pred, alpha=0.6, edgecolors='k', linewidths=0.3, s=30, color=color)
    ax.plot(lims, lims, 'k--', lw=1.5)
    ax.set_xlabel('Actual Respiration')
    ax.set_ylabel('Predicted Respiration')
    ax.set_title(f'{label} Model (Test)\nR² = {r2:.3f}  |  RMSE = {rmse:.4f}')
    ax.spines[['top', 'right']].set_visible(False)

plt.suptitle('Baseline vs Tuned Model — Test Set Predictions',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Step 6: SHAP Analysis — Explaining Model Predictions

Feature importance (Step 4) tells us *which* features matter to the model, but not **how** or **in what direction**. SHAP solves this.

### What is a SHAP value?
**SHAP** (SHapley Additive exPlanations) assigns each feature a contribution value
for every individual prediction, grounded in game theory (Shapley values):

- A **positive SHAP value** means the feature *increased* the predicted respiration
- A **negative SHAP value** means the feature *decreased* the prediction
- SHAP values are in the **same units as the target** (ppm CO₂-C)
- They are **additive**: the sum of all SHAP values equals the model's deviation from the dataset mean

### Plots
1. **Beeswarm (summary) plot** — overview of all features and all test samples; shows distribution, magnitude, and direction of effects simultaneously
2. **Dependence plot** — zooms into one feature to show how its value affects predictions (with optional interaction colouring)
3. **Bar plot** — mean absolute SHAP values as a direction-aware importance ranking

> **Note:** We use the **baseline model** (`rf`) from Step 4. If you completed Step 5, replace `rf` with `rf_tuned`.

In [ ]:
!pip install shap

In [ ]:
import shap

# TreeExplainer is optimised for tree-based models — fast and exact
explainer   = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_test)

print(f"SHAP values shape : {shap_values.shape}")
print(f"  rows = test samples ({X_test.shape[0]})")
print(f"  cols = features     ({X_test.shape[1]})")

### 6.1 Beeswarm Summary Plot

Each **dot** is one test sample. The x-axis is the SHAP value (contribution to prediction).
Colour encodes the **feature value** (red = high, blue = low).

**How to read it:**
- Rows are sorted by mean |SHAP| (most important feature at the top)
- A cluster of **red dots on the right** means high feature values *increase* respiration
- A cluster of **blue dots on the right** means low feature values *increase* respiration

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
shap.summary_plot(shap_values, X_test, feature_names=feature_cols, show=False)
plt.title(
    'SHAP Beeswarm Plot\n'
    'Each dot = one test sample; colour = feature value (red=high, blue=low)',
    fontsize=10,
)
plt.tight_layout()
plt.show()

### 6.2 Dependence Plot for the Most Important Feature

A dependence plot shows the **relationship between one feature's value and its SHAP value**
across all test samples. The colour encodes an automatically selected interaction feature.

- The **x-axis** is the raw feature value
- The **y-axis** is the SHAP value (impact on prediction)
- **Vertical spread** at a given x-value reveals interactions with the colour-coded feature

In [ ]:
# Select the top feature by mean absolute SHAP value
mean_abs_shap = np.abs(shap_values).mean(axis=0)
top_feat_idx  = int(np.argmax(mean_abs_shap))
top_feat_name = feature_cols[top_feat_idx]

print(f"Top feature by mean |SHAP| : {top_feat_name}")

shap.dependence_plot(
    top_feat_idx, shap_values, X_test,
    feature_names=feature_cols,
    show=False,
)
plt.title(
    f'SHAP Dependence Plot: {top_feat_name}\n'
    '(colour = auto-selected interaction feature)',
    fontsize=10,
)
plt.tight_layout()
plt.show()

### 6.3 Global Importance — Mean |SHAP|

This bar chart ranks features by **mean absolute SHAP value** — a direction-aware
alternative to the MDI importance from Step 4.

Compare the two rankings:
- Features that appear **high in both** are robustly important
- Features ranked differently may have complex interactions that MDI misses

In [ ]:
shap.summary_plot(
    shap_values, X_test,
    feature_names=feature_cols,
    plot_type='bar',
    show=False,
)
plt.title('Mean |SHAP Value| per Feature\n(direction-aware importance ranking)', fontsize=10)
plt.tight_layout()
plt.show()